In [5]:

!pip install -q google-genai pandas openpyxl


In [6]:

import os
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass(
    "Enter your Gemini API key: "
)


Enter your Gemini API key: ··········


In [8]:

# =====================================================
# 3. CREATE GEMINI CLIENT
# =====================================================

from google import genai

client = genai.Client(
    api_key=os.environ["GEMINI_API_KEY"]
)

print("Gemini client connected successfully")


Gemini client connected successfully


In [9]:

# =====================================================
# 4. UPLOAD EXCEL FILE
# =====================================================

from google.colab import files

uploaded = files.upload()

print("Excel file uploaded successfully")


Saving gemini_nl2sql_sales_data_5000.xlsx to gemini_nl2sql_sales_data_5000.xlsx
Excel file uploaded successfully


In [10]:

# =====================================================
# 5. READ EXCEL DATA
# =====================================================

import pandas as pd

file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name)

print("Excel file loaded successfully")

print("\nFirst 5 rows:")
display(df.head())

print("\nTotal rows:", len(df))

print("\nColumn names:")
print(df.columns.tolist())


Excel file loaded successfully

First 5 rows:


,SaleID,ProductName,Category,City,Quantity,Amount,SaleDate,Status
0,1,Mobile,Electronics,Coimbatore,1,60598,2025-10-09,Completed
1,2,Headphones,Accessories,Chennai,12,19668,2026-11-24,Cancelled
2,3,Speaker,Accessories,Coimbatore,7,8820,2025-01-31,Completed
3,4,Tablet,Electronics,Coimbatore,9,490077,2025-01-28,Cancelled
4,5,Webcam,Accessories,Bengaluru,7,21035,2026-04-05,Cancelled



Total rows: 5000

Column names:
['SaleID', 'ProductName', 'Category', 'City', 'Quantity', 'Amount', 'SaleDate', 'Status']


In [11]:

# =====================================================
# 6. CREATE SQLITE DATABASE
# =====================================================

import sqlite3

conn = sqlite3.connect("sales_nl2sql.db")

cursor = conn.cursor()

print("SQLite database created successfully")


SQLite database created successfully


In [12]:

# =====================================================
# 7. INSERT EXCEL DATA INTO SQLITE
# =====================================================

df.to_sql(
    "sales",
    conn,
    if_exists="replace",
    index=False
)

conn.commit()

print("Excel data inserted into SQLite successfully")
print("Total records:", cursor.execute(
    "SELECT COUNT(*) FROM sales"
).fetchone()[0])


Excel data inserted into SQLite successfully
Total records: 5000


In [13]:

# =====================================================
# 8. GET DATABASE SCHEMA
# =====================================================

cursor.execute("PRAGMA table_info(sales)")

schema_rows = cursor.fetchall()

schema = "TABLE: sales\nCOLUMNS:\n"

for row in schema_rows:
    column_name = row[1]
    data_type = row[2]

    schema += f"{column_name} {data_type}\n"

print(schema)


TABLE: sales
COLUMNS:
SaleID INTEGER
ProductName TEXT
Category TEXT
City TEXT
Quantity INTEGER
Amount INTEGER
SaleDate TEXT
Status TEXT



In [23]:

# =====================================================
# 9. SQL VALIDATION FUNCTION
# =====================================================

def validate_sql(sql):

    sql_upper = sql.strip().upper()

    if not sql_upper.startswith("SELECT"):
        return False

    dangerous_commands = [
        "INSERT",
        "UPDATE",
        "DELETE",
        "DROP",
        "ALTER",
        "CREATE",
        "REPLACE",
        "ATTACH",
        "DETACH"
    ]

    for command in dangerous_commands:
        if command in sql_upper:
            return False

    return True


print("SQL validation function created")


SQL validation function created


In [24]:

# =====================================================
# 10. GENERATE SQL USING GEMINI
# =====================================================

def generate_sql(question):

    prompt = f"""
You are an expert SQLite SQL generator.

Convert the user's natural language question
into a SQLite SELECT query.

DATABASE SCHEMA:

{schema}

RULES:

1. Generate only SELECT queries.
2. Use only the sales table.
3. Use only the columns provided in the schema.
4. Do not generate INSERT, UPDATE, DELETE,
   DROP, ALTER, CREATE, REPLACE,
   ATTACH or DETACH.
5. Return only the SQL query.
6. Do not provide explanations.
7. Do not use markdown code fences.

USER QUESTION:

{question}
"""

    response = client.models.generate_content(
        model="models/gemini-3.8-flash",
        contents=prompt
    )

    sql = response.text.strip()

    return sql


print("Gemini SQL generation function created")


Gemini SQL generation function created


In [16]:

# =====================================================
# 11. EXECUTE GENERATED SQL
# =====================================================

def execute_sql(sql):

    print("\nGENERATED SQL")
    print("-" * 50)
    print(sql)

    if not validate_sql(sql):
        print("\nUnsafe SQL rejected.")
        return

    try:
        cursor.execute(sql)

        results = cursor.fetchall()

        print("\nQUERY RESULT")
        print("-" * 50)

        if results:
            for row in results:
                print(row)
        else:
            print("No records found.")

    except sqlite3.Error as e:
        print("\nSQLite Error:", e)


print("SQL execution function created")


SQL execution function created


In [17]:

# =====================================================
# 12. ASK NATURAL LANGUAGE QUESTION
# =====================================================

question = input(
    "Ask a question about your sales data: "
)

print("\nYour question:")
print(question)


Ask a question about your sales data: What is the sales amount?

Your question:
What is the sales amount?


In [27]:

# =====================================================
# 13. SEND QUESTION TO GEMINI
# =====================================================

generated_sql = generate_sql(question)

print("SQL generated successfully")
print(generated_sql)


SQL generated successfully
SELECT SUM(Amount) FROM sales;


In [21]:

# =====================================================
# 14. CHECK AVAILABLE GEMINI MODELS
# =====================================================

for model_info in client.models.list():

    model_name = model_info.name
    supported_actions = getattr(
        model_info,
        "supported_actions",
        []
    )

    if "generateContent" in supported_actions:

        print(model_name)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemi

In [28]:

# =====================================================
# 15. VALIDATE AND EXECUTE SQL
# =====================================================

execute_sql(generated_sql)



GENERATED SQL
--------------------------------------------------
SELECT SUM(Amount) FROM sales;

QUERY RESULT
--------------------------------------------------
(619440483,)


In [29]:

# =====================================================
# 16. CLOSE DATABASE
# =====================================================

conn.close()

print("Database connection closed successfully")


Database connection closed successfully
